In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score , mean_squared_error , mean_absolute_error
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor , GradientBoostingRegressor
from lightgbm import LGBMRegressor

In [2]:
all_data = pd.read_csv(r'../data/processed/all_data.csv')

In [3]:
all_data.shape

(135060, 11)

In [4]:
from sklearn.model_selection import train_test_split
X = all_data.drop(columns=['price_in_USD'])
y = all_data['price_in_USD']
X_train , X_test , y_train , y_test = train_test_split(X , y , test_size=0.2 , random_state=42 , shuffle=True)

print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(108048, 10)
(27012, 10)
(108048,)
(27012,)


In [5]:
X_train

,country,location,building_construction_year,building_total_floors,apartment_floor,apartment_rooms,apartment_bedrooms,apartment_bathrooms,apartment_total_area,apartment_living_area
125397,Montenegro,"Budva, Bar, Bar Municipality, Montenegro",2005.0,6.0,5.0,4.0,3.0,1.0,130.0,110.5
6995,Spain,"Valencian Community, Costa Calida, Spain",2006.5,4.5,5.0,4.0,4.0,2.5,173.0,105.0
89510,Spain,"Mil Palmeras, Orihuela, Valencian Community, e...",2018.0,5.0,1.0,4.0,3.0,2.0,101.0,60.0
82350,Spain,"Torrevieja, Valencian Community, el Baix Segur...",2022.0,8.0,2.0,3.0,2.0,1.5,112.0,61.0
57946,Belarus,"Homel Region, Homel, Belarus",1986.0,9.0,9.0,2.5,2.0,1.5,65.0,26.0
...,...,...,...,...,...,...,...,...,...,...
110268,Hungary,"Great Plain and North, Debreceni jaras, Hajdú-...",2021.0,3.0,5.5,3.5,4.0,1.0,200.0,172.5
119879,Czech Republic,"Prague, Czech Republic",2012.0,2.0,2.5,3.0,2.0,1.0,56.0,36.0
103694,Croatia,"Velika Gorica, City of Velika Gorica, Zagreb C...",1978.0,1.0,1.0,1.0,4.0,2.5,255.0,142.0
131932,Turkey,"Mahmutlar, Mediterranean Region, Alanya, Turkey",1978.5,11.0,3.0,2.0,1.0,1.0,70.0,64.5


## Target Encoding

In [6]:
from category_encoders import TargetEncoder
encoder = TargetEncoder(min_samples_leaf=5 , smoothing=10)
encoder.fit(X_train[['country' , 'location']], y_train)
X_train_encoded = encoder.transform(X_train[['country' , 'location']])
X_test_encoded  = encoder.transform(X_test[['country' , 'location']])

In [7]:
X_train_encoded

,country,location
125397,514842.470625,435879.722356
6995,691462.950309,293534.872239
89510,691462.950309,437700.428393
82350,691462.950309,257107.015891
57946,71635.572400,42734.758045
...,...,...
110268,202523.475311,191391.090535
119879,278704.759588,342089.984127
103694,604180.865107,320406.576660
131932,374312.309839,231236.367466


In [8]:
X_train.drop(columns=['country' , 'location'] , inplace=True , axis=1)
X_test.drop(columns=['country' , 'location'] , inplace=True , axis=1)

In [9]:
X_train_all = pd.concat([X_train , X_train_encoded] , axis=1)
X_test_all  = pd.concat([X_test , X_test_encoded] , axis=1)

In [10]:
X_train_all

,building_construction_year,building_total_floors,apartment_floor,apartment_rooms,apartment_bedrooms,apartment_bathrooms,apartment_total_area,apartment_living_area,country,location
125397,2005.0,6.0,5.0,4.0,3.0,1.0,130.0,110.5,514842.470625,435879.722356
6995,2006.5,4.5,5.0,4.0,4.0,2.5,173.0,105.0,691462.950309,293534.872239
89510,2018.0,5.0,1.0,4.0,3.0,2.0,101.0,60.0,691462.950309,437700.428393
82350,2022.0,8.0,2.0,3.0,2.0,1.5,112.0,61.0,691462.950309,257107.015891
57946,1986.0,9.0,9.0,2.5,2.0,1.5,65.0,26.0,71635.572400,42734.758045
...,...,...,...,...,...,...,...,...,...,...
110268,2021.0,3.0,5.5,3.5,4.0,1.0,200.0,172.5,202523.475311,191391.090535
119879,2012.0,2.0,2.5,3.0,2.0,1.0,56.0,36.0,278704.759588,342089.984127
103694,1978.0,1.0,1.0,1.0,4.0,2.5,255.0,142.0,604180.865107,320406.576660
131932,1978.5,11.0,3.0,2.0,1.0,1.0,70.0,64.5,374312.309839,231236.367466


## Apply log transform

In [11]:
y_train_log = np.log1p(y_train)
y_test_log  = np.log1p(y_test)

In [348]:
train_encoded = pd.concat([X_train_all , y_train_log] , axis=1)
train_encoded.to_csv(r'../data/processed/train_encoded.csv' , index=False)

In [349]:
test_encoded = pd.concat([X_test_all , y_test_log] , axis=1)
test_encoded.to_csv(r'../data/processed/test_encoded.csv' , index=False)

In [351]:
all_data_encoded = pd.concat([train_encoded , test_encoded], axis=0)
all_data_encoded.to_csv(r'../data/processed/all_data_encoded.csv' , index=False)

## Gradient Boosting

In [347]:
gbr = GradientBoostingRegressor(
    n_estimators=1200,      
    learning_rate=0.02,     
    max_depth=5,           
    min_samples_split=4,
    min_samples_leaf=2,
    subsample=0.8,         
    max_features="sqrt",    
    random_state=42
)
gbr.fit(X_train_all, y_train_log)
y_train_pred_gbr = gbr.predict(X_train_all)
y_test_pred_gbr  = gbr.predict(X_test_all)
print(f"Gradient Boosting Regressor Train")
print(f"R² Train : {r2_score(y_train_log, y_train_pred_gbr)}")
print(f"Gradient Boosting Regressor Test")
print(f"R² Test  : {r2_score(y_test_log,  y_test_pred_gbr)}")

Gradient Boosting Regressor Train
R² Train : 0.8201231577343945
Gradient Boosting Regressor Test
R² Test  : 0.7772050728049134


## XGBoosting

In [ ]:
xgb = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective="reg:squarederror",
    tree_method="hist"
)

xgb.fit(X_train_all, y_train_log)

y_train_pred_log = xgb.predict(X_train_all)
y_test_pred_log  = xgb.predict(X_test_all)

mse_train = mean_squared_error(y_train_log, y_train_pred_log)
rmse_train = np.sqrt(mse_train)
r2_train = r2_score(y_train_log, y_train_pred_log)

mse_test = mean_squared_error(y_test_log, y_test_pred_log)
rmse_test = np.sqrt(mse_test)
r2_test = r2_score(y_test_log, y_test_pred_log)

print("XGBoost Train Metrics")
print("Train MSE :", mse_train)
print("Train RMSE:", rmse_train)
print("Train R²  :", r2_train)

print("\nXGBoost Test Metrics")
print("Test MSE :", mse_test)
print("Test RMSE:", rmse_test)
print("Test R²  :", r2_test)

===== XGBoost Train Metrics =====
Train MSE : 0.1916318213803999
Train RMSE: 0.4377577199552281
Train R²  : 0.8419250110474726

===== XGBoost Test Metrics =====
Test MSE : 0.27056227824777335
Test RMSE: 0.5201560133726931
Test R²  : 0.7817396360779805


## CatBoost

In [343]:
cat_model = CatBoostRegressor(
    depth=8,
    learning_rate=0.03,
    n_estimators=2000,
    subsample=0.8,
    colsample_bylevel=0.8,
    l2_leaf_reg=5,
    loss_function="RMSE",
    random_state=42,
    verbose=200
)
cat_model.fit(X_train_all, y_train_log)

y_train_pred = cat_model.predict(X_train_all)
y_test_pred  = cat_model.predict(X_test_all)


mse_train = mean_squared_error(y_train_log, y_train_pred)
rmse_train = np.sqrt(mse_train)
r2_train = r2_score(y_train_log, y_train_pred)

mse_test = mean_squared_error(y_test_log, y_test_pred)
rmse_test = np.sqrt(mse_test)
r2_test = r2_score(y_test_log, y_test_pred)

print("CatBoost Train Metrics")
print("Train MSE :", mse_train)
print("Train RMSE:", rmse_train)
print("Train R²  :", r2_train)

print("\nCatBoost Test Metrics")
print("Test MSE :", mse_test)
print("Test RMSE:", rmse_test)
print("Test R²  :", r2_test)

0:	learn: 1.0796179	total: 91.7ms	remaining: 3m 3s
200:	learn: 0.5139923	total: 8.67s	remaining: 1m 17s
400:	learn: 0.4909733	total: 19.8s	remaining: 1m 18s
600:	learn: 0.4776980	total: 30.8s	remaining: 1m 11s
800:	learn: 0.4680186	total: 45.2s	remaining: 1m 7s
1000:	learn: 0.4607645	total: 55.9s	remaining: 55.8s
1200:	learn: 0.4544905	total: 1m 8s	remaining: 45.6s
1400:	learn: 0.4491078	total: 1m 23s	remaining: 35.5s
1600:	learn: 0.4448052	total: 1m 36s	remaining: 24s
1800:	learn: 0.4408997	total: 1m 53s	remaining: 12.6s
1999:	learn: 0.4370287	total: 2m 10s	remaining: 0us
CatBoost Train Metrics
Train MSE : 0.19099412745234287
Train RMSE: 0.43702874899981453
Train R²  : 0.8424510377788713

CatBoost Test Metrics
Test MSE : 0.2682350808337328
Test RMSE: 0.5179141635770669
Test R²  : 0.7836169670858223


## LightGBM

In [345]:
lgb_model = LGBMRegressor(
    n_estimators=1500,
    learning_rate=0.03,
    num_leaves=64,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=5,
    reg_alpha=3,
    random_state=42
)

lgb_model.fit(X_train_all, y_train_log)

y_train_pred_log = lgb_model.predict(X_train_all)
y_test_pred_log  = lgb_model.predict(X_test_all)



mse_train = mean_squared_error(y_train_log, y_train_pred_log)
rmse_train = np.sqrt(mse_train)
r2_train = r2_score(y_train_log, y_train_pred_log)

mse_test = mean_squared_error(y_test_log, y_test_pred_log)
rmse_test = np.sqrt(mse_test)
r2_test = r2_score(y_test_log, y_test_pred_log)


print("LightGBM Train Metrics")
print("Train MSE :", mse_train)
print("Train RMSE:", rmse_train)
print("Train R²  :", r2_train)

print("\nLightGBM Test Metrics ")
print("Test MSE :", mse_test)
print("Test RMSE:", rmse_test)
print("Test R²  :", r2_test)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.020182 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1629
[LightGBM] [Info] Number of data points in the train set: 108048, number of used features: 10
[LightGBM] [Info] Start training from score 12.238022
LightGBM Train Metrics
Train MSE : 0.16555709678874178
Train RMSE: 0.4068870811278502
Train R²  : 0.8634337655542965

LightGBM Test Metrics 
Test MSE : 0.2591783907466769
Test RMSE: 0.5090956597209182
Test R²  : 0.7909229244688376


## Random Forset

In [346]:
rf = RandomForestRegressor(
    n_estimators=600,
    max_depth=20,
    min_samples_split=4,
    min_samples_leaf=2,
    max_features='sqrt',
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_all, y_train_log)

y_train_pred_log = rf.predict(X_train_all)
y_test_pred_log  = rf.predict(X_test_all)

mse_train = mean_squared_error(y_train_log, y_train_pred_log)
rmse_train = np.sqrt(mse_train)
r2_train = r2_score(y_train_log, y_train_pred_log)

mse_test = mean_squared_error(y_test_log, y_test_pred_log)
rmse_test = np.sqrt(mse_test)
r2_test = r2_score(y_test_log, y_test_pred_log)

print("Random Forest Train Metrics ")
print("Train MSE :", mse_train)
print("Train RMSE:", rmse_train)
print("Train R²  :", r2_train)

print("\n Random Forest Test Metrics")
print("Test MSE :", mse_test)
print("Test RMSE:", rmse_test)
print("Test R²  :", r2_test)

Random Forest Train Metrics 
Train MSE : 0.10094751656329867
Train RMSE: 0.3177223891438856
Train R²  : 0.9167295000872928

 Random Forest Test Metrics
Test MSE : 0.2624031030393778
Test RMSE: 0.5122529678190042
Test R²  : 0.788321575592317


In [12]:
import joblib
joblib.dump(encoder, "../artifacts/target_encoder.pkl")

['../artifacts/target_encoder.pkl']